# Tutorial 0: Onboarding Roadmap

## What is Bayesian metamodeling?

Imagine you have four separate models of early T-cell receptor signaling — one for membrane geometry, one for CD45 segregation, one for Lck activity, one for TCR phosphorylation. Each is independently parameterized and reasonably good in its domain, but they don't talk to each other. **Bayesian metamodeling is how you make them agree, propagate uncertainty across them, and produce a *joint posterior* over what all four are saying together.** This curriculum walks you from a 9-point toy sweep to that point in 9 tutorials.

The framework is **spec-driven** (one typed JSON contract per model), **CLI-first** (`bayesmm validate / plan / run / surrogate / meta`), and treats every run as a reproducible experiment with full provenance. The destination of this curriculum is `projects/tcr_signaling/` — the read-only submodule reproducing Neve-Oz, Sherman & Raveh (Frontiers in Immunology, 2024). You'll see those four models again in Tutorial 9.


## What T0 gives you

T0 is orientation, not practice — you won't run a model here. By the end of this
notebook you will be able to:

1. **Say what a metamodel is** in one sentence, and why four separate T-cell models
   need one.
2. **Name the five artifacts** that flow through every tutorial: spec → DOE plan →
   sweep → surrogate → metamodel. (The diagram further down is the whole curriculum
   on one line.)
3. **Tell whether your kernel can run a given tutorial**, using `bayesmm doctor`,
   without guessing at environment names.
4. **Decide what to install** — and, just as usefully, what *not* to: the backends
   are per-tutorial, not prerequisites for starting.

If you only remember one thing from T0, make it the data-flow diagram. Every later
tutorial is a zoom-in on one arrow of it.

## Environment first (simple rule)

Use a Jupyter kernel that already belongs to the conda (or venv) environment you
want. That kernel environment is what persists across all notebook cells.

You don't need to memorize environment names — the preflight cell below runs
**`bayesmm doctor`**, which reports your actual OS, Python, environment, and which
optional backends (PyMC / SBI) are installed. If something is missing, run
**`bayesmm setup`** for platform-correct install commands.

No installation commands run automatically in this notebook.


## Vocabulary you'll see across these tutorials

Definitions are intentionally one-sentence each. Later tutorials reference these terms; if you're confused mid-tutorial, come back here.

- **Spec**: a typed JSON contract describing one model — its identity, IO schema, runner, adapter, DOE plan, and storage. Validated by `bayesmm validate`.
- **DOE** (*design of experiments*): the set of input points at which you'll run the model. Two strategies in this curriculum: `grid` (cartesian product) and `sobol` (deterministic space-filling).
- **Sweep**: one execution of a model across all DOE points. Produces a centralized `sweep_rows.csv` with one row per point.
- **Adapter**: the small piece of code that turns a DOE point into a process invocation (e.g. `python_cli_adapter_v1`, `biomodels_sbml_adapter_v1`).
- **Surrogate**: a fast probabilistic model fit to a sweep's outputs that lets you predict at arbitrary inputs without re-running the simulator. Two backends here: PyMC GP and SBI NPE.
- **Coupling**: a constraint linking variables across models. **Deterministic** coupling propagates a value (`z = α·C + β`); **probabilistic** coupling propagates a distribution (e.g. `y ~ Normal(C, σ)`). The two have qualitatively different effects on the joint.
- **Metamodel**: the composed object that wires multiple surrogates together via couplings and produces a joint posterior when sampled.
- **Joint posterior**: the multi-variable distribution the metamodel produces when sampled — what your coupled models believe *together*, including the correlations the couplings induce.
- **Posterior predictive**: the surrogate's belief about the model output at a new input, expressed as a distribution (mean + width), not a point estimate. The width is the lesson.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent

os.chdir(project_root)
src_path = project_root / "src"
if src_path.is_dir() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

env = os.environ.copy()
src_str = str(src_path)
# os.pathsep is ":" on POSIX and ";" on Windows — never hardcode the separator.
env["PYTHONPATH"] = (
    src_str
    if not env.get("PYTHONPATH")
    else os.pathsep.join([src_str, env["PYTHONPATH"]])
)

print(f"Project root: {project_root}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

# CLI preflight — confirms the package is importable in this kernel.
result = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"],
    cwd=project_root,
    env=env,
    text=True,
    capture_output=True,
)
if result.returncode != 0:
    raise RuntimeError(
        "CLI preflight failed. Make sure this notebook runs in a prepared environment.\n"
        f"stderr:\n{result.stderr}"
    )
print(result.stdout.strip())

# Environment diagnostic — `bayesmm doctor` reports OS, Python, conda/venv, and
# which optional backends (pymc / sbi / torch) are available. This is the fastest
# way to confirm your kernel is ready for the tutorials below; if a backend is
# missing, run `bayesmm setup` for platform-correct install commands.
doctor = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "doctor"],
    cwd=project_root,
    env=env,
    text=True,
    capture_output=True,
)
print()
print(doctor.stdout.strip() or doctor.stderr.strip())


Project root: /Users/barakraveh/Git/metamodeler_codex_scaffold_docs
Python executable: /Users/barakraveh/miniconda3/envs/py312_bayesmm_sbi/bin/python3.12
Python version: 3.12.12


bayesian-metamodeling 0.1.0



Bayesian Metamodeling environment diagnostic
OS:        Darwin 25.4.0 (arm64)
Python:    CPython 3.12.12 (/Users/barakraveh/miniconda3/envs/py312_bayesmm_sbi/bin/python3.12)
Env:       conda (py312_bayesmm_sbi) -> /Users/barakraveh/miniconda3/envs/py312_bayesmm_sbi
Tools:     pip=26.0.1, conda=conda 26.1.1
Shell:     zsh (interactive=False)
Repo root: /Users/barakraveh/Git/metamodeler_codex_scaffold_docs (writable=True)

Optional backends
----------------------------------------
  [OK]   arviz    0.23.4
  [OK]   pymc     5.28.1
  [OK]   sbi      0.25.0
  [OK]   torch    2.5.1


## What you need to install (and what's framework vs tutorial)

The framework `bayesian_metamodeling` is intentionally lean. Tutorials add their own infrastructure on top. **Three categories** to keep straight, ordered from "always" to "tutorial-2-only":

| Category | What it is | When you need it | Install (pip) |
|---|---|---|---|
| **Framework runtime** | `pydantic`, `requests`, `scipy` | Always — these come with `pip install -e .` | `pip install -e .` |
| **Optional surrogate backends** | `[pymc]` (T5/T7/T8/T9) and `[sbi]` (T6) | Only for tutorials that use that backend | `pip install -e ".[pymc,sbi]"` |
| **Tutorial infrastructure** | `[tutorials]` = `jupyter`, `matplotlib`, `numpy` | To execute the notebooks themselves | `pip install -e ".[tutorials]"` |
| **BioModels SBML simulator** | `[biomodels]` = `libroadrunner`, `tellurium` | **Tutorial 2 only** — to run published BioModels SBML | `pip install -e ".[biomodels]"` |

**The most common gotcha**: `pip install -e ".[pymc,sbi]"` (the README's recommendation) gives you the surrogate backends but NOT `libroadrunner`/`tellurium`. Tutorial 2 will then skip its actual SBML run with a preflight banner. That's fine if you want to skip T2 — install `[biomodels]` only when you're ready to do it.

**One-shot for the impatient**: `pip install -e ".[all]"` installs everything (both backends + tutorial infrastructure + BioModels simulator). Larger download but skips the per-tutorial install dance.

> **About env-name hardcoding in install commands.** Pip's `install` operates on whatever Python it was invoked with — it doesn't care about conda env names. So the commands above are env-agnostic: activate the env running this notebook's kernel, then run them. The package-status cell below prints the kernel env path so you can verify which env is active. The bootstrap cells in T2 (and any tutorial that needs an optional dep) print the same env path before suggesting an install, so the install command always references the right env *for the reader*.

> **About conda for these specific deps.** `pymc` / `arviz` / `pytorch` / `sbi` are on conda-forge and `conda env create -f environment.yml -n py312_bayesmm` (recommended starting point for new lab members) installs all framework + backend + tutorial deps in one shot. **`libroadrunner` and `tellurium` are PyPI-only — not on conda-forge as of 2026.** Even inside a conda env, install them with `pip install libroadrunner tellurium`. The `environment.yml` deliberately excludes them so the conda solve stays fast.

### Per-tutorial dependency map

| # | Needs `[pymc]` | Needs `[sbi]` | Needs `[biomodels]` | Notes |
|---|:-:|:-:|:-:|---|
| T1 | — | — | — | Pure CLI loop on a toy model |
| T2 | — | — | **YES** | BioModels SBML run; preflight skips cleanly if not installed |
| T3 | — | — | — | Spec validation only — no model execution |
| T4 | — | — | — | DOE planning + the toy run |
| T5 | **YES** | — | — | PyMC GP surrogate fit/eval |
| T6 | — | **YES** | — | SBI NPE surrogate fit/eval |
| T7 | **YES** | — | — | Two-model coupling (uses pre-built example surrogates) |
| T8 | **YES** | — | — | Three-model coupling |
| T9 | **YES** | — | — | Capstone — student fits a PyMC surrogate |

The package status cell below shows what's installed in *this* kernel, categorized the same way.


In [2]:
## NOTHING in this cell runs automatically — these are reference commands.
## Pick the line(s) that match what you want to do, paste into a terminal,
## ACTIVATE the env running this notebook's kernel first, then run.
## Restart the Jupyter kernel after installing so cached imports refresh.
##
## ----- Framework runtime (always required; usually already done) -----
#   pip install -e .
##
## ----- Tutorial infrastructure (required to execute these notebooks) -----
#   pip install -e ".[tutorials]"          # jupyter + matplotlib + numpy
##
## ----- Optional surrogate backends (per the per-tutorial map above) -----
#   pip install -e ".[pymc]"               # for T5, T7, T8, T9
#   pip install -e ".[sbi]"                # for T6
##   conda alternatives (these ARE on conda-forge):
#   conda install -c conda-forge pymc arviz
#   conda install -c conda-forge pytorch sbi
##
## ----- Tutorial 2 only: BioModels SBML simulator -----
##   IMPORTANT: libroadrunner and tellurium are PyPI-only, NOT on conda-forge.
##   Use pip even inside a conda env.
#   pip install -e ".[biomodels]"          # libroadrunner + tellurium
#   # or equivalently:
#   pip install libroadrunner tellurium
##
## ----- One-shot: everything for every tutorial -----
#   pip install -e ".[all]"
##
## ----- Or use the conda env file (recommended for new lab members) -----
##   `environment.yml` covers framework + both backends + tutorial deps.
##   It does NOT include the BioModels simulator (libroadrunner/tellurium
##   are PyPI-only — see above), so add them with pip if you want T2 to
##   actually execute the SBML model.
#   conda env create -f environment.yml -n <pick a name>
#   conda activate <pick a name>
#   pip install libroadrunner tellurium     # only if you want T2 end-to-end
##
## `bayesmm setup` generates a platform-correct version of the above for
## *your* OS + shell. Run it (uncommented) if you want the framework to pick:
#   python -m bayesian_metamodeling.cli.main setup


## Package status (in THIS kernel, categorized)

Reports what's installed in the active Jupyter kernel, grouped by the install matrix above. Anything `not_installed` in a category you need is a clear next step — refer to the install commands in the previous cell.


In [ ]:
import importlib.metadata as md
import importlib.util
import sys


def _check(pkg_name: str, import_name: str | None = None) -> str:
    spec_name = import_name or pkg_name
    if importlib.util.find_spec(spec_name) is None:
        return "not_installed"
    try:
        return md.version(pkg_name)
    except md.PackageNotFoundError:
        return "installed (version unknown)"


# Categorized check matching the install matrix above.
groups = {
    "Framework runtime": [
        ("bayesian-metamodeling", "bayesian_metamodeling"),
        ("pydantic", None),
        ("requests", None),
        ("scipy", None),
    ],
    "Tutorial infrastructure": [
        ("jupyter", None),
        ("nbclient", None),
        ("nbconvert", None),
        ("notebook", None),
        ("matplotlib", None),
        ("numpy", None),
    ],
    "Optional backend: PyMC (needed for T5/T7/T8/T9)": [
        ("pymc", None),
        ("arviz", None),
    ],
    "Optional backend: SBI (needed for T6)": [
        ("torch", None),
        ("sbi", None),
    ],
    "Tutorial 2 only: BioModels SBML simulator": [
        ("libroadrunner", "roadrunner"),
        ("tellurium", None),
    ],
}

print(f"Kernel env : {sys.prefix}")
print(f"Python     : {sys.version.split()[0]}")
print()
for group_name, pkgs in groups.items():
    print(f"  {group_name}")
    for pkg_name, import_name in pkgs:
        status = _check(pkg_name, import_name)
        marker = "OK  " if status != "not_installed" else "MISS"
        print(f"    [{marker}] {pkg_name:<25} {status}")
    print()


## Tutorial map

Each tutorial answers one question and leaves you with one new capability. Don't worry about the time estimates — they're rough.

| # | What you build | What you'll be able to do |
|---|---|---|
| 1 | Run a 9-point toy sweep, plot two heatmaps from one centralized CSV | Drive the `validate → plan → run` CLI loop end-to-end |
| 2 | Run a real published BioModels SBML model with a dense `k_on` sweep | Recognize the spec contract is the same shape for toy and real models |
| 3 | Break a spec on purpose, then read and fix the validator output | Read a Pydantic validation error and fix the spec without trial-and-error |
| 4 | Compare `grid` vs `sobol` DOE strategies on the same toy | Pick the right DOE strategy for your model's dimensionality |
| 5 | Fit a PyMC GP surrogate, evaluate it on new inputs with uncertainty | Read a posterior-predictive width and decide whether to trust a prediction |
| 6 | Fit an SBI NPE surrogate on the same data, compare to PyMC | Tell when the two backends agree (and what disagreement means) |
| 7 | Couple two surrogates with `equality_soft`, sample the joint | See what "agreement between models" looks like as a scatter cloud |
| 8 | Chain three surrogates, observe uncertainty propagation along the chain | Budget noise across a coupled cascade |
| 9 | Compose your own DOE → surrogate → metamodel, write a 3-sentence report | Drive the framework end-to-end without copy-paste — the capstone |

After T9, the natural next step is `projects/tcr_signaling/` — the same workflow at full scale on four real biological models.


## How the pieces fit together

This is the data flow each tutorial slots into. Tutorials 1-2 cover the left side (spec → run → CSV). Tutorials 3-4 are about specs and DOE planning. Tutorials 5-6 are the middle (CSV → surrogate). Tutorials 7-8 are the right side (surrogates → coupling → joint metamodel). Tutorial 9 composes the whole thing end-to-end.

```
┌────────┐   ┌──────────┐   ┌────────────┐   ┌──────────────────┐   ┌─────────────┐   ┌──────────────┐
│  Spec  │ → │ DOE plan │ → │   Sweep    │ → │ sweep_rows.csv   │ → │  Surrogate  │ → │   Metamodel  │
│ (JSON) │   │  (grid/  │   │ (executes  │   │ (centralized,    │   │  artifact   │   │  + couplings │
│        │   │   sobol) │   │  N points) │   │  one row/point)  │   │ (.artifact) │   │              │
└────────┘   └──────────┘   └────────────┘   └──────────────────┘   └─────────────┘   └──────────────┘
   T3            T4              T1, T2                                  T5, T6               T7, T8
                                                                                                │
                                                                                                ▼
                                                                                      ┌──────────────────┐
                                                                                      │  Joint posterior │
                                                                                      │ (samples_dataset │
                                                                                      │     .json)       │
                                                                                      └──────────────────┘
                                                                                              T9
```

Three storage artifacts you'll see across the tutorials: **`sweep_rows.csv`** (one per sweep, the canonical handoff to surrogates), **`.artifact.json`** (one per fitted surrogate, contains the trained model + provenance), and **`samples_dataset.json`** (one per metamodel sampling run, contains posterior draws).


## Tips for new lab members

- **Keep a short run log**: commands, run IDs, and a one-line interpretation per major step. The framework's run registry tracks IDs, but your interpretation notes are what make a sweep reproducible six months later.
- **If a dependency is missing, continue with fallback paths.** If `bayesmm doctor` reports SBI missing and you don't plan to do T6, that's fine — skip T6 for now. If `tellurium`/`libroadrunner` are missing, T2 will skip its `bayesmm run` cell with a clear banner — you can still inspect the spec and plan output. Come back when you have time to install.
- **`MM_BIOMODELS_OFFLINE=1` for sandboxed runs.** If you're on a machine without internet (or you don't want to download a 1.4 MB SBML on the spot), set this env var and T2's adapter will refuse to download — it'll either use a cached SBML you already have or print a `curl` recipe so you can pre-populate the cache from a different machine.
- **Inspect one artifact file after each major command.** `cat tmp/run_registry.json | head` after a sweep, `ls tmp/surrogate_artifacts/*/artifact.json` after a fit. Looking at the artifact once teaches you more than reading the docs about it.
- **Time sinks to know about**: T2's first run downloads ~1.4 MB from BioModels (5-30 s). T5 and T6's surrogate fits take 10-60 s each. T8's joint sampling takes 30-90 s on a CPU. Everything else is sub-second.
- **The destination is `projects/tcr_signaling/`.** When this curriculum starts feeling like toy work, that's the signal you're ready to read its README.


## Before you move to T1

A 60-second self-check. If you can't answer these from what you just read, scroll
back — they're the load-bearing ideas, and T1 assumes all four.

1. **What does a spec contain, and what validates it?**
2. **Where does a sweep put its results, and in what format?**
   *(This one matters most: every surrogate in T5–T9 trains from that file.)*
3. **Which tutorials need PyMC, which needs SBI, and which needs neither?**
4. **Your kernel right now — can it run T5?** If you don't know, re-read the
   package-status output above rather than finding out mid-T5.

**Answers:** (1) identity, IO schema, runner, adapter, DOE plan and storage;
`bayesmm validate`. (2) `sweep_rows.csv`, one row per DOE point, under the run
store. (3) PyMC → T5, T7, T8, T9; SBI → T6; neither → T1, T3, T4. (4) The status
cell says `pymc: installed` or `not_installed`.

Onward to **T1**, where you drive the whole loop on a 9-point toy model that runs
in seconds — small enough that you can check every number by hand.

## Final check: the kernel is wired correctly

This cell asserts the bootstrap + diagnostics ran cleanly. If it fails, T1+ won't run either — fix the env first.

In [ ]:
# Self-check: bootstrap printed env info; bayesmm doctor exited 0 (run during bootstrap).
# Failures here mean the kernel can't import the package or the CLI is broken.
import importlib.util as _u
assert _u.find_spec("bayesian_metamodeling") is not None, (
    "bayesian_metamodeling package not importable in this kernel — "
    "see install matrix above."
)
import subprocess as _sp
import sys as _s
_doctor = _sp.run(
    [_s.executable, "-m", "bayesian_metamodeling.cli.main", "doctor"],
    capture_output=True, text=True, timeout=30,
)
assert _doctor.returncode == 0, f"`bayesmm doctor` exit {_doctor.returncode}: {_doctor.stderr[-200:]}"
print(f"\n[T0 self-check OK] kernel: {_s.prefix}")
